In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import yaml


input_shape_st = (60, 900)

CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10uncertainties_dict
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))



def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model


         


In [ ]:
mne.set_log_level(verbose='CRITICAL')

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-small',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
load_dir  = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_power"

In [ ]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
#phase_perturbationss = [0.2,0.5,2,3,5,10]
#phase_peturbations = np.arange(45, 316, 45)

amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]

In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, ch_names

In [ ]:


def load_distances(subject_index, factors, rep=1):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_distance"
    
    distances = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        distances[band_name] = {}

        for factor in factors:
            file_path = f"parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir, file_path)
            distances[band_name][factor] = np.load(load_path, allow_pickle=True).item()

    return distances

In [ ]:
def calculate_diff_per_channel_dw(pred_label_original, freq_bands, amplification_factors, ch_names, distances, subject_index=2, rep=1, take_abs=False):
    dir_constrained = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_power"
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir_constrained, file_path)
            perturbed_data = np.load(load_path, allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = np.array(perturbed_data[ch_name])
                if take_abs:
                    diff = np.abs(pred_label_original - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])
                else:
                    diff = (pred_label_original - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

In [ ]:
def plot_PDP_per_channel_and_freq_band(pred_label_original, freq_bands, phase_perturbations, ch_names, distances, abs_diff=False, subject_index=2, rep=1):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        fig, axs = plt.subplots(ncols=3, nrows=20, figsize=(20, 40), sharex=True)
        fig.tight_layout()
        fig.suptitle(f'Amplitude Difference per Channel for {band_name}')
        for ch_idx,ch_name in enumerate(ch_names):
            mean_diffs = []
            median_diffs = []
            for factor_idx,factor in enumerate(phase_perturbations):
                file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
                load_path = os.path.join(load_dir, file_path)
                perturbed_data = np.load(load_path, allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                else:
                    diff = (pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                mean_diffs.append(np.mean(diff))
                #median_diffs.append(np.median(diff))
            
            axs[ch_idx//3, ch_idx%3].plot(phase_perturbations, mean_diffs, label='Mean', color='blue')
            axs[ch_idx//3, ch_idx%3].set_xticks(phase_perturbations)
            axs[ch_idx//3, ch_idx%3].set_xticklabels(phase_perturbations)
            #ax.scatter(phase_perturbations, median_diffs, label='Median', color='red')
            axs[ch_idx//3, ch_idx%3].set_title(f'{ch_name}')
            if ch_idx%3 >=57:
                axs[ch_idx//3, ch_idx%3].set_xlabel('Amplification Factor')
            if ch_idx%3 == 0:
                if abs_diff:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Amplitude Difference')
            
            #ax.legend()
           

def plot_top_10_channels(pred_label_original, freq_bands, phase_perturbations, ch_names, top_channels_median, distances, abs_diff=False, subject_index=2, rep=1):
    fig, axs = plt.subplots(ncols=2, nrows=5, figsize=(13, 8), sharex=True, gridspec_kw={'hspace': 0.3})
    #fig.tight_layout()
    fig.suptitle(f'Top 10 Channels with Largest Amplitude Difference {subject_index}')
    
    for band_name, (low_freq, high_freq) in freq_bands.items():
        top_10_channels = top_channels_median[band_name][1.5]  # Assuming factor 2 is used to determine top 10 channels
        for ch_idx, ch_name in enumerate(top_10_channels):
            mean_diffs = []
            for factor in phase_perturbations:
                file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
                load_path = os.path.join(load_dir, file_path)
                perturbed_data = np.load(load_path, allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                else:
                    diff = (pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                mean_diffs.append(np.mean(diff))
                    
            axs[ch_idx//2, ch_idx%2].plot(phase_perturbations, mean_diffs, label=f'{band_name} Band')
            axs[ch_idx//2, ch_idx%2].set_xticks(phase_perturbations)
            axs[ch_idx//2, ch_idx%2].set_xticklabels(phase_perturbations)
            axs[ch_idx//2, ch_idx%2].set_title(f'{ch_name}')
            if ch_idx%2 == 0:
                if abs_diff:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Amplitude Difference')
            if ch_idx//2 == 4:
                axs[ch_idx//2, ch_idx%2].set_xlabel('Amplification Factor')
    
    for ax in axs.flat:
        ax.legend()
    fig.savefig("top_10_channels_amplitude_difference_behavior.png")
    plt.show()
    

#plot_top_10_channels(original_prediction, freq_bands, phase_peturbations, ch_names, top_channels_median, abs_diff=False)


In [ ]:
def create_index_groups(uncertainties, subject_index, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < len(uncertainties):
        end = min(start + group_size, len(uncertainties)-20)
        
        index_group = np.zeros(len(uncertainties), dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    
    index_groups_all[subject_index] = index_groups_subject
    return index_groups_all

In [ ]:
def get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, phase_peturbations, top_k=10):
    top_channels_mean = {}
    top_channels_median = {}
    prediction_diff_mean = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        top_channels_mean[band_name] = {}
        top_channels_median[band_name] = {}
        prediction_diff_mean[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in phase_peturbations:
            mean_diffs = mean_diff_per_channel[band_name][factor]
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Sort the channels based on their differences
            sorted_mean_diffs = sorted(mean_diffs.items(), key=lambda item: item[1], reverse=True)
            sorted_median_diffs = sorted(median_diffs.items(), key=lambda item: item[1], reverse=True)
            
            # Select the top k channels
            top_channels_mean[band_name][factor] = [ch for ch, _ in sorted_mean_diffs[:top_k]]
            top_channels_median[band_name][factor] = [ch for ch, _ in sorted_median_diffs[:top_k]]
            
            # Store the prediction differences for the top k channels
            prediction_diff_mean[band_name][factor] = {ch: mean_diffs[ch] for ch in top_channels_mean[band_name][factor]}
            prediction_diff_median[band_name][factor] = {ch: median_diffs[ch] for ch in top_channels_median[band_name][factor]}
    
    return top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median

#top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands,#phase_peturbations)

In [ ]:
def get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, phase_peturbations, top_k=10, index_group=None, reverese=True):
    """
    Get the top channels with highest differences for each frequency band and phase perturbation.
    
    Parameters:
    -----------
    mean_diff_per_channel : dict
        Dictionary with mean differences per channel
    median_diff_per_channel : dict
        Dictionary with median differences per channel
    freq_bands : dict
        Dictionary of frequency bands
    phase_peturbations : array-like
        Array of phase perturbation values
    top_k : int, default=10
        Number of top channels to return
    index_group : array-like, default=None
        Boolean array for filtering data. If None, all data is used.
    
    Returns:
    --------
    tuple
        (top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median)
    """
    top_channels_mean = {}
    top_channels_median = {}
    prediction_diff_mean = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        top_channels_mean[band_name] = {}
        top_channels_median[band_name] = {}
        prediction_diff_mean[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in phase_peturbations:
            # Apply index_group filtering if provided
            if index_group is not None:
                # Extract values for the specified indices
                mean_diffs = {ch: np.mean(np.abs(mean_diff_per_channel[band_name][factor][ch][index_group])) 
                             for ch in mean_diff_per_channel[band_name][factor]}
                
                median_diffs = {ch: np.median(np.abs(median_diff_per_channel[band_name][factor][ch][index_group])) 
                               for ch in median_diff_per_channel[band_name][factor]}
            else:
                mean_diffs = mean_diff_per_channel[band_name][factor]
                median_diffs = median_diff_per_channel[band_name][factor]
            
            # Sort the channels based on their differences
            sorted_mean_diffs = sorted(mean_diffs.items(), key=lambda item: item[1], reverse=reverese)
            sorted_median_diffs = sorted(median_diffs.items(), key=lambda item: item[1], reverse=reverese)
            
            # Select the top k channels
            top_channels_mean[band_name][factor] = [ch for ch, _ in sorted_mean_diffs[:top_k]]
            top_channels_median[band_name][factor] = [ch for ch, _ in sorted_median_diffs[:top_k]]
            
            # Store the prediction differences for the top k channels
            prediction_diff_mean[band_name][factor] = {ch: mean_diffs[ch] for ch in top_channels_mean[band_name][factor]}
            prediction_diff_median[band_name][factor] = {ch: median_diffs[ch] for ch in top_channels_median[band_name][factor]}
    
    return top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median

In [ ]:
def plot_top_10_ICE(pred_label_original, freq_bands, phase_peturbations, ch_names, top_channels_median, distances,
                    abs_diff=False, index_group=0, ax=None, band_name="alpha", subject_index=2, i=0, rep=1):
    if ax is None:
        fig, axs = plt.subplots(ncols=2, nrows=5, figsize=(16, 8), sharex=True)
        fig.tight_layout()
        fig.suptitle(f'ICE for Top 10 Channels - {band_name} Band')
    else:
        axs = ax

    top_10_channels = top_channels_median[band_name][1.5]  # Assuming factor 2 is used to determine top 10 channels
    ICE_array = np.zeros((len(top_10_channels), len(phase_peturbations), len(pred_label_original[index_group])))

    for ch_idx, ch_name in enumerate(top_10_channels):
        for factor_idx, factor in enumerate(phase_peturbations):
            file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(load_dir, file_path)
            perturbed_data = np.load(load_path, allow_pickle=True).item()
                               
            perturbed_amplitude = np.array(perturbed_data[ch_name])[index_group]
            
            # Calculate difference from original prediction
            if abs_diff:
                diff = np.abs(pred_label_original[index_group] - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])[index_group]
            else:
                diff = (pred_label_original[index_group] - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])[index_group]
                
            ICE_array[ch_idx, factor_idx] = diff

        # Randomly sample 20 ICE curves to plot
        sampled_indices = np.random.choice(len(pred_label_original[index_group]), size=20, replace=True)
        for idx in sampled_indices:
            if ax is None:
                axs[ch_idx//2, ch_idx%2].plot(phase_peturbations, ICE_array[ch_idx, :, idx], color='#03A89E', alpha=0.3)
            else:
                axs[ch_idx].plot(phase_peturbations, ICE_array[ch_idx, :, idx], color='#03A89E', alpha=0.3)
        if ax is None:
            axs[ch_idx//2, ch_idx%2].plot(phase_peturbations, ICE_array[ch_idx].mean(axis=1), color='#FF6103')
            axs[ch_idx//2, ch_idx%2].set_xticks(phase_peturbations)
            axs[ch_idx//2, ch_idx%2].set_xticklabels(phase_peturbations)
            axs[ch_idx//2, ch_idx%2].set_title(f'{ch_name}')
            if ch_idx % 2 == 0:
                if abs_diff:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Amplitude Difference')
            if ch_idx // 2 == 4:
                axs[ch_idx//2, ch_idx%2].set_xlabel('Amplification Factor')
        else:
            axs[ch_idx].plot(phase_peturbations, ICE_array[ch_idx].mean(axis=1), color='#FF6103', label=f"mean {ICE_array[ch_idx].mean():.2f}, std = {ICE_array[ch_idx].std(axis=1).mean():.2f}")
            #axs[ch_idx].legend(fontsize='13', loc='upper center')
            axs[ch_idx].set_xticks(phase_peturbations)
            axs[ch_idx].set_xticklabels(phase_peturbations, rotation=45, fontsize='14')
            #axs[ch_idx].set_title(f'{ch_name}')
            if abs_diff:
                if i == 0:
                    axs[ch_idx].set_ylabel('Abs Diff')
            else:
                if i == 0:
                    axs[ch_idx].set_ylabel("$\Delta p_w$", fontsize='17')

            if ch_idx == len(top_10_channels) - 1:
                axs[ch_idx].set_xlabel('Amplification Factor')
        if ax is None:
            fig.savefig(f"ICE_top_10_channels_{band_name}.png")

In [ ]:
def plot_averaged_channel_effects(pred_label_original, freq_bands, phase_peturbations, ch_names, distances, abs_diff=False, subject_index=2,rep=1):
    """
    Plot the average effect of phase perturbations across all channels for each frequency band.
    
    Parameters:
    -----------
    pred_label_original : array-like
        The original prediction
    freq_bands : dict
        Dictionary of frequency bands with their range
    phase_peturbations : array-like
        Array of phase perturbation values
    ch_names : list
        List of channel names
    abs_diff : bool, default=False
        Whether to compute the absolute difference
    subject_index : int, default=2
        Subject index for loading the data
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    
    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diffs_all_channels = []
        
        for factor in phase_peturbations:
            channel_means = []
            
            for ch_name in ch_names:
                file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
                load_path = os.path.join(load_dir, file_path)
                perturbed_data = np.load(load_path, allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name]
                
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                else:
                    diff = (pred_label_original - perturbed_amplitude)/distances[band_name][factor][ch_name]
                    
                channel_means.append(np.mean(diff))
            
            mean_diffs_all_channels.append(np.mean(channel_means))
        
        ax.plot(phase_peturbations, mean_diffs_all_channels, marker='o', label=f'{band_name} ({low_freq}-{high_freq} Hz)')
    
    ax.set_xlabel('Phase Shift (degrees)', fontsize='x-large')
    if abs_diff:
        ax.set_ylabel('Mean Absolute Amplitude Difference', fontsize='x-large')
    else:
        ax.set_ylabel('Mean Amplitude Difference', fontsize='x-large')
    
    ax.set_title(f'Average Effect of Phase Perturbations Across All Channels - Subject {subject_index}', fontsize='x-large')
    ax.legend(fontsize='large')
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(f"average_phase_perturbation_effect_subject_{subject_index}.png")
    plt.show()
    
    return mean_diffs_all_channels



In [ ]:
cfg = load_config()

# PDP per freq-band for top 10 channels

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    original_prediction, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index, rep=1)
    distances = load_distances(subject_index, amplification_factors, rep=1)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors, ch_names, distances, subject_index=subject_index, rep=1)
    top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors)
    plot_top_10_channels(original_prediction, freq_bands, amplification_factors, ch_names, top_channels_median, distances, abs_diff=False, subject_index=subject_index)
    #create_index_groups(uncertainties, cfg.dataset.test_subject_indices, group_size=100)
    #mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_prediction, freq_bands, phase_peturbations, subject_index=subject_index)

# effect averaged over all channels

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    original_prediction, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index, rep=1)
    distances = load_distances(subject_index, amplification_factors, rep=1)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors, ch_names, distances, subject_index=subject_index)
    top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors)
    plot_averaged_channel_effects(original_prediction, freq_bands, amplification_factors, ch_names, distances,abs_diff=False, subject_index=subject_index)
    #create_index_groups(uncertainties, cfg.dataset.test_subject_indices, group_size=100)
    #mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_prediction, freq_bands, phase_peturbations, subject_index=subject_index)

# ICE for TOP 10 channels

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    original_prediction,  ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index, rep=1)
    distances = load_distances(subject_index, amplification_factors,rep=1)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors, ch_names, distances, subject_index=subject_index,rep=1, take_abs=True)
    top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=1)
    #plot_top_10_channels(original_prediction, freq_bands, phase_peturbations, ch_names, top_channels_median, abs_diff=False, subject_index=subject_index)
    index_groups_all = create_index_groups(original_prediction, subject_index, group_size=100)
    fig, axs = plt.subplots(ncols=len(index_groups_all[subject_index]), nrows=1, figsize=(16,4), sharex=True, sharey=True)  
    fig.tight_layout(rect=[0.02,0.1,1,0.9])
    fig.suptitle(f'PDP-ICE for Top channel ({top_channels_median["gamma"][1.5][0]}), Subject {subject_index}', fontsize='20')
    
    for i, time_group in enumerate(index_groups_all[subject_index]):
        index_groups_all = plot_top_10_ICE(original_prediction, freq_bands, amplification_factors, ch_names, top_channels_median, distances, abs_diff=False, index_group=time_group, band_name="gamma", ax=[axs[i]], subject_index=subject_index, i=i)
    fig.savefig(f"power_ICE_top_10_channels_gamma_subject_{subject_index}.png", dpi=300, bbox_inches='tight')


In [ ]:

original_prediction,  ch_names = load_predicted_amplitude_for_subject(subject_index=45, rep=1)
distances = load_distances(45, amplification_factors,rep=1)
mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors, ch_names, distances, subject_index=45,rep=1)
top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=1)
    #plot_top_10_channels(original_prediction, freq_bands, phase_peturbations, ch_names, top_channels_median, abs_diff=False, subject_index=subject_index)
index_groups_all = create_index_groups(original_prediction, 45, group_size=100)
fig, axs = plt.subplots(ncols=len(index_groups_all[45]), nrows=1, figsize=(16,4), sharex=True, sharey=True)  
fig.tight_layout(rect=[0.02,0.1,1,0.9])
fig.suptitle(f'PDP-ICE for Top channel ({top_channels_median["gamma"][1.5][0]}), Subject {45}', fontsize='20')
    
for i, time_group in enumerate(index_groups_all[45]):
    index_groups_all = plot_top_10_ICE(original_prediction, freq_bands, amplification_factors, ch_names, top_channels_median, distances, abs_diff=False, index_group=time_group, band_name="gamma", ax=[axs[i]], subject_index=45, i=i)
fig.savefig(f"power_ICE_top_10_channels_gamma_subject_{45}.png", dpi=300, bbox_inches='tight')

In [ ]:

original_prediction,  ch_names = load_predicted_amplitude_for_subject(subject_index=80, rep=1)
distances = load_distances(80, amplification_factors,rep=1)
mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors, ch_names, distances, subject_index=80,rep=1)
top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=1)
    #plot_top_10_channels(original_prediction, freq_bands, phase_peturbations, ch_names, top_channels_median, abs_diff=False, subject_index=subject_index)
index_groups_all = create_index_groups(original_prediction, 80, group_size=100)
fig, axs = plt.subplots(ncols=len(index_groups_all[80]), nrows=1, figsize=(16,4), sharex=True, sharey=True)  
fig.tight_layout(rect=[0.02,0.1,1,0.9])
fig.suptitle(f'PDP-ICE for Top channel ({top_channels_median["gamma"][1.5][0]}), Subject {80}', fontsize='20')
    
for i, time_group in enumerate(index_groups_all[80]):
    index_groups_all = plot_top_10_ICE(original_prediction, freq_bands, amplification_factors, ch_names, top_channels_median, distances, abs_diff=False, index_group=time_group, band_name="gamma", ax=[axs[i]], subject_index=80, i=i)
fig.savefig(f"power_ICE_top_10_channels_gamma_subject_{80}.png", dpi=300, bbox_inches='tight')

In [ ]:
def topoplots_progression(subject_index=2, band_name="gamma", factor=1.5, take_abs=True, ticklims=None, rep=1, joint_colorbar=False):
    # Load data
    file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
    load_path = os.path.join(load_dir, file_path)
    perturbed_data = np.load(load_path, allow_pickle=True).item()
    distances = load_distances(subject_index, amplification_factors, rep=1)                       
    
    # Load original prediction and get channel info
    original_prediction, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
    all_perturbed_preds = np.zeros((60, len(original_prediction)))
    all_perturbed_dists = np.zeros((60, len(original_prediction)))
    for ch_idx, ch_name in enumerate(ch_names):
        all_perturbed_preds[ch_idx] = perturbed_data[ch_name]
        all_perturbed_dists[ch_idx] = distances[band_name][factor][ch_name]

    # Get subject channel info
    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info

    index_groups_all = create_index_groups(original_prediction, subject_index, group_size=100)

    fig, axes = plt.subplots(ncols=len(index_groups_all[subject_index]), nrows=1, figsize=(16, 5), sharex=True, sharey=True)
    fig.tight_layout(rect=[-0.01, -0.01, 1.01, 1.5])
    
    # Pre-calculate all importances if using joint colorbar
    if joint_colorbar:
        all_importances = []
        for index_group in index_groups_all[subject_index]:
            if take_abs:
                diff = np.abs(original_prediction[index_group] - all_perturbed_preds[:, index_group])/all_perturbed_dists[:, index_group]
            else:
                diff = (original_prediction[index_group] - all_perturbed_preds[:, index_group])/all_perturbed_dists[:, index_group]
            
            n_samples = diff.shape[1]
            channel_importance = np.median(diff/n_samples, axis=1)
            all_importances.append(channel_importance)
        
        # Get global min and max for consistent scaling
        vmin = min(np.min(imp) for imp in all_importances)
        vmax = max(np.max(imp) for imp in all_importances)

        if np.abs(vmin) > np.abs(vmax):
            vmax = np.abs(vmin)
        else:
            vmin = -np.abs(vmax)
    topos = []  # Store topomap objects for joint colorbar
    for i, index_group in enumerate(index_groups_all[subject_index]):
        channel_importances = np.zeros(len(ch_names))
        if take_abs:
            diff = np.abs(original_prediction[index_group] - all_perturbed_preds[:, index_group])/all_perturbed_dists[:, index_group]
        else:
            diff = (original_prediction[index_group] - all_perturbed_preds[:, index_group])/all_perturbed_dists[:, index_group]

        n_samples = diff.shape[1]
        channel_importances += np.median(diff/n_samples, axis=1)

        # Use global min/max if joint colorbar is requested
        if joint_colorbar:
            topo, _ = mne.viz.plot_topomap(channel_importances, info_subj, axes=axes[i], show=False, 
                                         names=ch_names, vlim=(vmin,vmax))
            topos.append(topo)
        else:
            topo, _ = mne.viz.plot_topomap(channel_importances, info_subj, axes=axes[i], show=False, names=ch_names)
            
            # Add individual colorbar if not using joint
            cb = plt.colorbar(topo, ax=axes[i], location='bottom', pad=0.05, shrink=0.95)
            cb.set_label('Median Difference', fontsize=14)
            cb.ax.tick_params(labelsize=14)
            cb.ax.set_xticklabels(['{:.4f}'.format(x) for x in cb.get_ticks()])
            
            if ticklims is not None:
                vmin = channel_importances.min()
                vmax = channel_importances.max()
                cb.set_ticks([vmin, (vmin+vmax)/2, vmax])
                cb.ax.set_xticklabels(['{:.2e}'.format(x) for x in cb.get_ticks()], rotation=22)
    
    # Add joint colorbar if requested
    if joint_colorbar:
        cbar_ax = fig.add_axes([0.2, 0.45, 0.6, 0.03])  # [left, bottom, width, height]
        cbar = fig.colorbar(topos[0], cax=cbar_ax, location="bottom")
        cbar.set_label('weighted $\Delta p$', fontsize=16)
        cbar.ax.tick_params(labelsize=14)
        if ticklims is not None:
            cbar.set_ticks([vmin, (vmin+vmax)/2, vmax])
            cbar.ax.set_xticklabels(['{:.2e}'.format(x) for x in cbar.get_ticks()], rotation=22)
    if joint_colorbar:
        fig.suptitle(f'{band_name.capitalize()} Band - Subject {subject_index} - Factor {factor}', fontsize=20, y=1.07)
    else:
        fig.suptitle(f'{band_name.capitalize()} Band - Subject {subject_index} - Factor {factor}', fontsize=18)
    fig.savefig(f"topo_progression_power_{band_name}_subject_{subject_index}.png", dpi=300, bbox_inches='tight')


cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    topoplots_progression(subject_index=subject_index, band_name="gamma", factor=1.5, take_abs=True, ticklims=None, rep=1, joint_colorbar=True)

In [ ]:
topoplots_progression(subject_index=45, band_name="gamma", factor=1.5, take_abs=False, ticklims=None, rep=1, joint_colorbar=True)

In [ ]:
topoplots_progression(subject_index=80, band_name="gamma", factor=1.5, take_abs=False, ticklims=False, rep=1)

In [ ]:
topoplots_progression(subject_index=67, band_name="gamma", factor=1.5, take_abs=False, ticklims=False, rep=1, joint_colorbar=True)

In [ ]:
topoplots_progression(subject_index=41, band_name="gamma", factor=1.5, take_abs=False, ticklims=False, rep=1, joint_colorbar=True)

In [ ]:
topoplots_progression(subject_index=62, band_name="gamma", factor=1.5, take_abs=False, ticklims=False, rep=1, joint_colorbar=True)

In [ ]:
topoplots_progression(subject_index=69, band_name="gamma", factor=1.5, take_abs=False, ticklims=True)

change in mean only seems significant for subset of channels ? perhabs those are the channels that were post important in the pretrained model?

# top 10 channel development over time

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    original_prediction, uncertainties,_, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
    distances = load_distances(subject_index, amplification_factors, rep=1)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel_dw(original_prediction, freq_bands, amplification_factors,subject_index=subject_index)
    index_groups_all = create_index_groups(uncertainties, subject_index, group_size=100)

    fig, axs = plt.subplots(ncols=len(index_groups_all[subject_index]), nrows=10, figsize=(15,12), sharex=True)
    fig.tight_layout()
    for i, time_group in enumerate(index_groups_all[subject_index]):
        top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, index_group=time_group)
        print(top_channels_median)

    #plot_top_10_channels(or
   

In [ ]:
subject_index = 2
original_prediction, uncertainties,_, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
index_groups_all = create_index_groups(uncertainties, subject_index, group_size=100)

#fig, axs = plt.subplots(ncols=len(index_groups_all[subject_index]), nrows=10, figsize=(15,12), sharex=True)
#fig.tight_layout()
for i, time_group in enumerate(index_groups_all[subject_index]):
    #print(time_group)
    mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_prediction, freq_bands, amplification_factors,subject_index=subject_index, index_group=time_group)
    top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=10)
    
    # Store the current time group's top channels in a dictionary for tracking changes over time
    if i == 0:
        # Initialize the dictionary to store top channels for each time group
        top_channels_over_time = {band: [] for band in freq_bands.keys()}
    
    # Add the current time group's top channels to the tracking dictionary
    for band in freq_bands.keys():
        top_channels_over_time[band].append(top_channels_median[band][2])
    
    # Print top channels for the current time group
    print(f"Time group {i} top channels:")
    for band in freq_bands.keys():
        print(f"  {band} band (180°): {top_channels_median[band][2][:3]}")  # Show just first 3 channels


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_channel_consistency_over_time(top_channels_over_time, freq_bands):
    """
    Plot the consistency of top channels over time segments for each frequency band.
    
    Parameters:
    -----------
    top_channels_over_time : dict
        Dictionary with frequency bands as keys and lists of top channels per time segment as values
    freq_bands : dict
        Dictionary of frequency bands with their frequency ranges
    """
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(15, 4*len(freq_bands)), sharex=True)
    
    # Get all unique channels across all time segments
    all_channels = set()
    for band in freq_bands.keys():
        for time_segment in top_channels_over_time[band]:
            all_channels.update(time_segment)
    all_channels = sorted(list(all_channels))
    
    for i, band_name in enumerate(freq_bands.keys()):
        # Create matrix to represent channel presence (rows=channels, columns=time segments)
        channel_presence = np.zeros((len(all_channels), len(top_channels_over_time[band_name])))
        
        # Fill the matrix with 1s where a channel appears in a time segment
        for t, time_segment_channels in enumerate(top_channels_over_time[band_name]):
            for j, channel in enumerate(all_channels):
                if channel in time_segment_channels:
                    channel_presence[j, t] = 1
        
        # Sort channels by consistency (most consistent at the top)
        consistency_scores = np.sum(channel_presence, axis=1)
        sorted_indices = np.argsort(consistency_scores)[::-1]
        sorted_channels = [all_channels[idx] for idx in sorted_indices]
        sorted_presence = channel_presence[sorted_indices]
        
        # Filter out channels that never appear
        active_channels = [ch for idx, ch in enumerate(sorted_channels) if consistency_scores[sorted_indices[idx]] > 0]
        active_presence = sorted_presence[:len(active_channels)]
        
        # Create heatmap
        ax = axes[i] if len(freq_bands) > 1 else axes
        im = ax.imshow(active_presence, cmap='viridis', aspect='auto', interpolation='none')
        ax.set_yticks(np.arange(len(active_channels)))
        ax.set_yticklabels(active_channels)
        ax.set_title(f'{band_name} ({freq_bands[band_name][0]}-{freq_bands[band_name][1]} Hz)')
        
        # Add time segment labels
        ax.set_xticks(np.arange(len(top_channels_over_time[band_name])))
        ax.set_xticklabels([f"T{t+1}" for t in range(len(top_channels_over_time[band_name]))])
        
        if i == len(freq_bands) - 1:
            ax.set_xlabel('Time Segment')
        
        # Add consistency score annotation
        for j in range(len(active_channels)):
            score = int(np.sum(active_presence[j]))
            ax.text(len(top_channels_over_time[band_name]) + 0.1, j, 
                     f"{score}/{len(top_channels_over_time[band_name])}", 
                     va='center', fontsize=10)
    
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.01, shrink=0.8)
        cbar.set_label('Channel Present in Top 10')
    
    plt.tight_layout()
    plt.savefig("top_channels_consistency_over_time.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    return active_channels, active_presence

# Call the function with the top_channels_over_time dictionary
active_channels, active_presence = plot_channel_consistency_over_time(top_channels_over_time, freq_bands)